In [17]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


In [16]:
from typing import TypedDict

class BatsmanState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int

    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [26]:
def calculate_sr(state: BatsmanState):
    sr = (state["runs"] / state["balls"]) * 100
    return {"sr": sr}

In [27]:
def calculate_bpb(state: BatsmanState):
    boundaries = state["fours"] + state["sixes"]

    bpb = state["balls"] / boundaries if boundaries else 0.0

    return {"bpb": bpb}

In [28]:
def calculate_boundary_percent(state: BatsmanState):
    boundary_runs = state["fours"] * 4 + state["sixes"] * 6

    boundary_percent = (
        (boundary_runs / state["runs"]) * 100
        if state["runs"] else 0.0
    )

    return {"boundary_percent": boundary_percent}

In [29]:
def summary(state: BatsmanState):
    summary_text = f"""
Strike Rate - {state['sr']:.2f}
Balls per boundary - {state['bpb']:.2f}
Boundary percent - {state['boundary_percent']:.2f}%
"""

    return {"summary": summary_text}

In [31]:

# Create the graph
graph = StateGraph(BatsmanState)

# Add nodes
graph.add_node("calculate_sr", calculate_sr)
graph.add_node("calculate_bpb", calculate_bpb)
graph.add_node(
    "calculate_boundary_percent",
    calculate_boundary_percent
)
graph.add_node("summary", summary)

# Parallel edges from START
graph.add_edge(START, "calculate_sr")
graph.add_edge(START, "calculate_bpb")
graph.add_edge(START, "calculate_boundary_percent")

# Join the parallel branches at summary
graph.add_edge("calculate_sr", "summary")
graph.add_edge("calculate_bpb", "summary")
graph.add_edge("calculate_boundary_percent", "summary")

# End the workflow
graph.add_edge("summary", END)

# Compile
workflow = graph.compile()


In [32]:
initial_state = {
    "runs": 78,
    "balls": 45,
    "fours": 8,
    "sixes": 4,
    "sr": 0.0,
    "bpb": 0.0,
    "boundary_percent": 0.0,
    "summary": ""
}

output = workflow.invoke(initial_state)

print(output["summary"])


Strike Rate - 173.33
Balls per boundary - 3.75
Boundary percent - 71.79%

